In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

class AIS_Advanced:
    def __init__(self, num_detectors=500, generations=60, start_mutation=0.3, end_mutation=0.01, random_seed=42):
        self.num_detectors = num_detectors
        self.generations = generations
        self.start_mutation = start_mutation
        self.end_mutation = end_mutation
        self.detectors = [] 
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=60) 
        self.rng = np.random.default_rng(random_seed)
        self.normal_memory = [] 

    def preprocess(self, X, train=True):
        if train:
            X_pca = self.pca.fit_transform(X)
            return self.scaler.fit_transform(X_pca)
        else:
            X_pca = self.pca.transform(X)
            return self.scaler.transform(X_pca)

    def fit(self, normal_data, disease_data):
        print(f"Evolution Phase: Training {self.num_detectors} Detectors")
        self.normal_memory = normal_data
        
        # 1. Initialize with specific Disease Samples (Cloning)
        seed_indices = self.rng.integers(0, len(disease_data), size=self.num_detectors)
        population = disease_data[seed_indices].copy()
        
        for gen in range(self.generations):
            # CALCULATE DYNAMIC MUTATION RATE (Linear Decay)
            current_mutation = self.start_mutation - ((self.start_mutation - self.end_mutation) * (gen / self.generations))
            
            # 2. Mutate
            noise = self.rng.normal(0, current_mutation, population.shape)
            mutants = population + noise
            
            # 3. Evaluate Fitness (Competitive)
            scores = []
            
            sim_to_mutants = cosine_similarity(disease_data, mutants)
            sim_to_normals = cosine_similarity(disease_data, self.normal_memory)
            
            # ROBUSTNESS: Instead of just max(), top 3 nearest normals are taken
            sim_to_normals_sorted = np.sort(sim_to_normals, axis=1)
            baseline_normal_sim = np.mean(sim_to_normals_sorted[:, -3:], axis=1) # Average of top 3
            
            for i in range(self.num_detectors):
                # Score
                margins = sim_to_mutants[:, i] - baseline_normal_sim
                
                positive_margins = margins[margins > 0]
                score = np.sum(positive_margins ** 2) 
                scores.append(score)
            
            scores = np.array(scores)
            
            # 4. Selection 
            best_indices = np.argsort(scores)[::-1]
            population = mutants[best_indices]
            
            if gen % 50 == 0:
                print(f"   Gen {gen}: Best Score={scores.max():.2f}, MutationRate={current_mutation:.4f}")
                
        self.detectors = population
        print("Evolution Complete.")

    def predict(self, samples, bias=1.02):
        # 1. Similarity to Detectors
        sim_to_detectors = cosine_similarity(samples, self.detectors)
        # Robustness: Take average of top 3 closest detectors
        sim_to_detectors_sorted = np.sort(sim_to_detectors, axis=1)
        score_detectors = np.mean(sim_to_detectors_sorted[:, -3:], axis=1)
        
        # 2. Similarity to Normals
        sim_to_normals = cosine_similarity(samples, self.normal_memory)
        # Robustness: Take average of top 3 closest normals
        sim_to_normals_sorted = np.sort(sim_to_normals, axis=1)
        score_normals = np.mean(sim_to_normals_sorted[:, -3:], axis=1)
        
        # 3. Decision
        # Bias > 1.0 reduces False Positives
        predictions = (score_detectors > (score_normals * bias)).astype(int)
        
        return predictions

try:
    df = pd.read_csv('GSE33000_Top10000_Var.csv', index_col=0)
except FileNotFoundError:
    print("Using Top5000 file...")
    df = pd.read_csv('GSE33000_Top5000_Var.csv', index_col=0)

labels = df['Diagnosis']
data = df.drop('Diagnosis', axis=1).values

normal_indices = np.where(labels.str.contains("C"))[0]
disease_indices = np.where(~labels.str.contains("C"))[0]

# Split (70/30)
n_split = int(len(normal_indices) * 0.7)
d_split = int(len(disease_indices) * 0.7)

train_normal_idx = normal_indices[:n_split]
test_normal_idx = normal_indices[n_split:]

train_disease_idx = disease_indices[:d_split]
test_disease_idx = disease_indices[d_split:]

X_train_normal = data[train_normal_idx]
X_train_disease = data[train_disease_idx]
X_test = data[np.concatenate([test_normal_idx, test_disease_idx])]
y_test = np.array([0]*len(test_normal_idx) + [1]*len(test_disease_idx))

print(f"Training: {len(X_train_normal)} Normal, {len(X_train_disease)} Disease")

ais = AIS_Advanced(
    num_detectors=1000,     
    generations=60,        
    start_mutation=0.3,    
    end_mutation=0.05     
)

# Preprocess
X_combined_train = np.vstack((X_train_normal, X_train_disease))
ais.preprocess(X_combined_train, train=True)

X_norm_scaled = ais.preprocess(X_train_normal, train=False)
X_dis_scaled = ais.preprocess(X_train_disease, train=False)

# Fit
ais.fit(X_norm_scaled, X_dis_scaled)

# Predict
# If False Positives are high -> Increase Bias
# If False Negatives are high -> Decrease Bias
X_test_scaled = ais.preprocess(X_test, train=False)
y_pred = ais.predict(X_test_scaled, bias=1.06)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nAccuracy: {acc:.2%}")
print("\nConfusion Matrix:")
print(f"True Normal:      {cm[0][0]}")
print(f"False Positive:   {cm[0][1]}")
print(f"False Negative:   {cm[1][0]}")
print(f"True Positive:    {cm[1][1]}")

In [ ]:
cross_df = pd.read_csv('GSE44770_Top10000_Var.csv', index_col=0)

cross_labels = cross_df['Diagnosis'].astype(str).str.strip()
cross_data = cross_df.drop('Diagnosis', axis=1).values

# print(cross_labels.value_counts().sort_index())
print(f"\nNormal (N): {(cross_labels == 'N').sum()}")
print(f"Alzheimer's (A): {(cross_labels == 'A').sum()}")

cross_X_test = cross_data
cross_y_test = np.array([0 if label == "A" else 1 for label in cross_labels])

cross_X_test_scaled = ais.preprocess(cross_X_test, train=False)
cross_y_pred = ais.predict(cross_X_test_scaled, bias=1.06)

cross_acc = accuracy_score(cross_y_test, cross_y_pred)
cross_cm = confusion_matrix(cross_y_test, cross_y_pred)

print(f"\nAccuracy: {cross_acc:.2%}")
print("\nConfusion Matrix:")
print(f"True Normal:      {cross_cm[0][0]}")
print(f"False Positive:   {cross_cm[0][1]}")
print(f"False Negative:   {cross_cm[1][0]}")
print(f"True Positive:    {cross_cm[1][1]}")